In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Car and Pedestrian Tracker

Modern CV Detection / Tracking Pipeline (April 2026)

Primary : YOLO26m (Ultralytics) for detection and tracking.
Export  : metrics.json with file-level detections + validation.json with output checks.
Data    : Auto-downloads demo files at runtime.


## Environment Setup


In [1]:
import os
# Define __file__ so save-path logic in the pipeline works correctly
__file__ = os.path.abspath('pipeline.py')
%matplotlib inline


## Dataset Setup

Datasets are **auto-downloaded** at runtime via the Kaggle API, sklearn, yfinance, or HuggingFace.

**Kaggle setup** (one-time): place your `kaggle.json` in `~/.kaggle/`.


In [2]:
# Kaggle API setup — credentials are read from ~/.kaggle/kaggle.json
# To create: https://www.kaggle.com/settings → API → Create New Token
import os, subprocess, sys
try:
    import kaggle  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
# Create project data & output directories
os.makedirs("data", exist_ok=True)
os.makedirs(os.path.join("outputs", "eda"), exist_ok=True)
os.makedirs(os.path.join("outputs", "models"), exist_ok=True)
os.makedirs(os.path.join("outputs", "results"), exist_ok=True)
print("Dataset directories ready")


C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Dataset directories ready


## Imports & Configuration


In [3]:
import os, json, time, warnings
from pathlib import Path
import urllib.request

warnings.filterwarnings("ignore")

TASK = "track"
SAVE_DIR = os.path.dirname(os.path.abspath(__file__))


## Download Samples


In [4]:
def download_samples():
    from pathlib import Path
    return [p for p in Path(SAVE_DIR).glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp", ".mp4", ".avi", ".mov")]


## Run Detection


In [5]:
def run_detection(files):
    from ultralytics import YOLO
    model = YOLO("yolo26m.pt")
    image_files = [f for f in files if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]
    out_dir = os.path.join(SAVE_DIR, "detections")
    os.makedirs(out_dir, exist_ok=True)
    metrics = {"model": "yolo26m", "task": "detect", "images": []}
    t0 = time.perf_counter()
    for f in image_files[:20]:
        results = model(str(f))
        for r in results:
            r.save(filename=os.path.join(out_dir, f.name))
            n_boxes = len(r.boxes) if r.boxes is not None else 0
            classes = [int(b.cls) for b in r.boxes] if r.boxes is not None else []
            metrics["images"].append({"file": f.name, "detections": n_boxes,
                                       "classes": dict(sorted({c: classes.count(c) for c in set(classes)}.items()))})
            if n_boxes:
                print(f"  {f.name}: {n_boxes} objects detected")
    elapsed = time.perf_counter() - t0
    metrics["time_s"] = round(elapsed, 1)
    metrics["total_images"] = len(metrics["images"])
    metrics["total_detections"] = sum(i["detections"] for i in metrics["images"])
    print(f"  Detection: {metrics['total_images']} images, {metrics['total_detections']} objects in {elapsed:.1f}s")
    print(f"  Results saved to {out_dir}")
    return metrics


## Run Tracking


In [6]:
def run_tracking(files):
    from ultralytics import YOLO
    model = YOLO("yolo26m.pt")
    video_files = [f for f in files if f.suffix in (".mp4", ".avi", ".mov")]
    if not video_files:
        print("  No video files found. Running detection on images instead.")
        return run_detection(files)
    metrics = {"model": "yolo26m", "task": "track", "videos": []}
    t0 = time.perf_counter()
    for v in video_files[:3]:
        model.track(str(v), persist=True, save=True, project=SAVE_DIR, name="tracking")
        metrics["videos"].append({"file": v.name})
        print(f"  Tracked: {v.name}")
    elapsed = time.perf_counter() - t0
    metrics["time_s"] = round(elapsed, 1)
    print(f"  Tracking: {len(metrics['videos'])} videos in {elapsed:.1f}s")
    return metrics


## Run Eda

Input file summary for detection.


In [7]:
def run_eda(files, save_dir):
    """Input file summary for detection."""
    print("\n" + "=" * 60)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 60)
    print(f"  Input files: {len(files)}")
    if files:
        total_size = sum(os.path.getsize(f) for f in files if os.path.isfile(f))
        print(f"  Total size: {total_size / 1024:.1f} KB")
    print("EDA complete.")


## Validate Results

Validate output coverage for detection / tracking demos.


In [8]:
def validate_results(metrics, files, save_dir):
    """Validate output coverage for detection / tracking demos."""
    validation = {
        "task": metrics.get("task", TASK),
        "input_files": len(files),
        "processed": int(metrics.get("total_images", len(metrics.get("videos", [])))),
        "time_s": round(float(metrics.get("time_s", 0)), 1),
    }
    if metrics.get("task") == "track":
        validation["processed"] = len(metrics.get("videos", []))
        validation["passed"] = validation["processed"] > 0 and validation["time_s"] >= 0
    else:
        validation["total_detections"] = int(metrics.get("total_detections", 0))
        validation["passed"] = validation["processed"] > 0 and validation["time_s"] >= 0
    out_path = os.path.join(save_dir, "validation.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(validation, f, indent=2)
    print(f"Validation saved to {out_path}")
    return validation


## Main Pipeline


In [9]:
def main():
    print("=" * 60)
    print(f"CV DETECTION | Task: {TASK} | Model: YOLO26m")
    print("=" * 60)
    files = download_samples()
    run_eda(files, SAVE_DIR)
    if TASK == "track":
        metrics = run_tracking(files)
    else:
        metrics = run_detection(files)
    metrics["validation"] = validate_results(metrics, files, SAVE_DIR)

    out_path = os.path.join(SAVE_DIR, "metrics.json")
    with open(out_path, "w") as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"Metrics saved to {out_path}")


## Execute Pipeline

Run the complete ML pipeline end-to-end:


In [10]:
main()


CV DETECTION | Task: track | Model: YOLO26m

EXPLORATORY DATA ANALYSIS
  Input files: 2
  Total size: 55884.1 KB
EDA complete.


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...


Using Python 3.13.12 environment at: C:\Users\ahmad\AppData\Local\Programs\Python\Python313
Resolved 2 packages in 1.08s
 Downloaded lap
Prepared 1 package in 360ms
Installed 1 package in 12ms
 + lap==0.5.13



requirements: AutoUpdate success  1.9s


WARNING requirements: Restart runtime or rerun command for updates to take effect



...

video 1/1 (frame 358/359) E:\Github\Machine-Learning-Projects\Computer Vision\Car and Pedestrian Tracker\pedestrain.mp4: 384x640 12 persons, 3 handbags, 29.7ms


video 1/1 (frame 359/359) E:\Github\Machine-Learning-Projects\Computer Vision\Car and Pedestrian Tracker\pedestrain.mp4: 384x640 14 persons, 3 handbags, 29.8ms


Speed: 1.9ms preprocess, 23.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)


Results saved to E:\Github\Machine-Learning-Projects\Computer Vision\Car and Pedestrian Tracker\tracking


  Tracked: pedestrain.mp4
  Tracking: 1 videos in 34.3s
Validation saved to E:\Github\Machine-Learning-Projects\Computer Vision\Car and Pedestrian Tracker\validation.json
Metrics saved to E:\Github\Machine-Learning-Projects\Computer Vision\Car and Pedestrian Tracker\metrics.json
